<a href="https://colab.research.google.com/github/sara010896/PPCA_UnB/blob/main/Lista_1_solucao_Sara_Borges.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lista 1 — Geração de números pseudo-aleatórios
**Métodos Computacionais Intensivos para Mineração de Dados** — Prof. Guilherme Rodrigues

Solução completa em **Python**. Aluno(a): _______________________

> Pré-requisitos: `numpy`, `matplotlib`. O arquivo `Dicionario.txt` deve estar no mesmo diretório.

In [ ]:
import os, unicodedata, re, numpy as np
import matplotlib
import matplotlib.pyplot as plt

os.makedirs("graficos", exist_ok=True)   # pasta para salvar as figuras

# Estilo idêntico ao usado nas figuras do relatório
plt.rcParams.update({
    "figure.dpi": 130, "font.size": 11, "axes.grid": True,
    "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False
})

# Paleta de cores do relatório
C1 = "#1f4e79"   # azul (estimativa)
C2 = "#c0504d"   # vermelho (valor teórico)
C3 = "#2e7d32"   # verde

rng = np.random.default_rng(2026)   # semente global -> reprodutibilidade

## Preparação: leitura e normalização do dicionário
Removemos acentos/cedilha (normalização NFKD), passamos para minúsculas e mantemos
apenas palavras com letras `a`–`z`. Guardamos o conjunto de palavras de 5 letras.

In [ ]:
def strip_accents(s):
    """Remove acentos/cedilha via decomposição Unicode (NFKD)."""
    return ''.join(c for c in unicodedata.normalize('NFKD', s)
                   if not unicodedata.combining(c))

palavras = []
with open('Dicionario.txt', encoding='utf-8') as f:
    for linha in f:
        w = strip_accents(linha.strip()).lower()
        if re.fullmatch(r'[a-z]+', w):
            palavras.append(w)

dic         = set(palavras)
dic5        = frozenset(w for w in dic if len(w) == 5)
ALFABETO    = np.array(list('abcdefghijklmnopqrstuvwxyz'))
N_VALIDAS_5 = len(dic5)
print("Palavras distintas:", len(dic))
print("Palavras de 5 letras:", N_VALIDAS_5)

# Questão 1 — Gerador de Babel
## Item (a) — Probabilidade de gerar palavra válida
Analítico: $p_a = |\mathcal{D}_5| / 26^5$.

In [ ]:
def gera_uniforme(n, k=5, rng=None):
    """Gera n sequências de k letras i.i.d. uniformes; retorna array de strings."""
    rng = np.random.default_rng() if rng is None else rng
    idx = rng.integers(0, 26, size=(n, k))
    return np.array([''.join(ALFABETO[linha]) for linha in idx])

N    = 300_000
seqs = gera_uniforme(N, rng=rng)
valida = np.fromiter((s in dic5 for s in seqs), dtype=bool, count=N)

p_a_teorico = N_VALIDAS_5 / 26**5          # 0.00045677
p_a_mc      = valida.mean()
print(f"p_a (Monte Carlo) = {p_a_mc:.6f}")
print(f"p_a (analítico)   = {p_a_teorico:.6f}")

n_acum = np.arange(1, N + 1)
estim  = np.cumsum(valida) / n_acum
erro_pad = np.sqrt(estim * (1 - estim) / n_acum)

# ---- Gráfico (idêntico ao relatório) ----
n_acum = np.arange(1, N + 1)
estim  = np.cumsum(valida) / n_acum
erro_pad = np.sqrt(estim * (1 - estim) / n_acum)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_acum, estim, color=C1, lw=1, label="Estimativa Monte Carlo")
ax.fill_between(n_acum, estim-1.96*erro_pad, estim+1.96*erro_pad, color=C1, alpha=.15, label="IC 95% aprox.")
ax.axhline(p_a_teorico, color=C2, ls="--", lw=1.6, label=f"Valor teórico = {p_a_teorico:.6f}")
ax.set_xscale("log")
ax.set_xlabel("Tamanho da amostra (escala log)"); ax.set_ylabel("Probabilidade estimada")
ax.set_title("Convergência da estimativa MC — P(palavra válida)")
ax.legend(frameon=False, fontsize=9)
ax.set_ylim(0, max(estim[100:].max()*1.4, p_a_teorico*2.5))
fig.tight_layout()
fig.savefig("graficos/Q1a_palavra_valida.png", dpi=130, bbox_inches="tight")
plt.show()

## Item (b) — Palíndromo
Analítico: $p_b = 26^3/26^5 = 1/26^2$ (independe de ser palavra válida).

In [ ]:
# Reaproveita as mesmas sequências do item (a) — eficiência computacional
palindromo = np.fromiter((s == s[::-1] for s in seqs), dtype=bool, count=N)
p_b_teorico = 1/26**2          # 0.00147929
p_b_mc      = palindromo.mean()
print(f"p_b (Monte Carlo) = {p_b_mc:.6f}")
print(f"p_b (analítico)   = {p_b_teorico:.6f}")

# ---- Gráfico (idêntico ao relatório) ----
n_acum = np.arange(1, N + 1)
runp   = np.cumsum(palindromo) / n_acum
sep    = np.sqrt(runp*(1-runp)/n_acum)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_acum, runp, color=C3, lw=1, label="Estimativa Monte Carlo")
ax.fill_between(n_acum, runp-1.96*sep, runp+1.96*sep, color=C3, alpha=.15, label="IC 95% aprox.")
ax.axhline(p_b_teorico, color=C2, ls="--", lw=1.6, label=f"Valor teórico = 1/26² = {p_b_teorico:.6f}")
ax.set_xscale("log")
ax.set_xlabel("Tamanho da amostra (escala log)"); ax.set_ylabel("Probabilidade estimada")
ax.set_title("Convergência da estimativa MC — P(palíndromo)")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("graficos/Q1b_palindromo.png", dpi=130, bbox_inches="tight")
plt.show()

## Item (c) — Gerador alternante consoante/vogal
1ª letra uniforme no alfabeto; demais alternam V/C. Todas as sequências geráveis são
equiprováveis, logo $p_c = (n_{VCVCV}+n_{CVCVC})/(26\cdot 5^2\cdot 21^2)$.

In [ ]:
VOGAIS     = list('aeiou')
CONSOANTES = [c for c in 'abcdefghijklmnopqrstuvwxyz' if c not in VOGAIS]

def padrao(w):
    return ''.join('V' if c in 'aeiou' else 'C' for c in w)

def gera_alternante(n, rng):
    """Primeira letra uniforme no alfabeto; demais alternam V/C."""
    res = np.empty((n, 5), dtype='<U1')
    primeiras = rng.integers(0, 26, n)
    for i, f in enumerate(primeiras):
        c0 = ALFABETO[f]; seq = [c0]; eh_vogal = c0 in 'aeiou'
        for _ in range(4):
            eh_vogal = not eh_vogal
            seq.append(VOGAIS[rng.integers(5)] if eh_vogal
                       else CONSOANTES[rng.integers(21)])
        res[i] = seq
    return np.array([''.join(r) for r in res])

# Contagem analítica
nV = sum(1 for w in dic5 if padrao(w) == 'VCVCV')   # 517
nC = sum(1 for w in dic5 if padrao(w) == 'CVCVC')   # 1474
p_c_teorico = (nV + nC) / (26 * 5**2 * 21**2)        # 0.0069458

g_alt  = gera_alternante(300_000, rng)
p_c_mc = np.mean([s in dic5 for s in g_alt])
print(f"n_VCVCV={nV}, n_CVCVC={nC}")
print(f"p_c (Monte Carlo) = {p_c_mc:.6f}")
print(f"p_c (analítico)   = {p_c_teorico:.6f}")

# ---- Gráfico (idêntico ao relatório) ----
vc   = np.fromiter((s in dic5 for s in g_alt), bool, len(g_alt))
nn   = np.arange(1, len(g_alt) + 1)
runc = np.cumsum(vc) / nn
sec  = np.sqrt(runc*(1-runc)/nn)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(nn, runc, color=C1, lw=1, label="Estimativa MC (gerador alternante)")
ax.fill_between(nn, runc-1.96*sec, runc+1.96*sec, color=C1, alpha=.15)
ax.axhline(p_c_teorico, color=C2, ls="--", lw=1.6, label=f"Valor teórico = {p_c_teorico:.5f}")
ax.set_xscale("log")
ax.set_xlabel("Tamanho da amostra (log)"); ax.set_ylabel("Probabilidade")
ax.set_title("Gerador alternante — P(palavra válida)")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("graficos/Q1c_gerador_alternante.png", dpi=130, bbox_inches="tight")
plt.show()

## Item (d) — Gerador ponderado por frequência, condicional a conter 'a'
$\Pr(\text{válida}\mid A)=\Pr(\text{válida}\cap A)/\Pr(A)$, com $\Pr(A)=1-(1-\pi_a)^5$.

In [ ]:
freq = {'a':14.63,'b':1.04,'c':3.88,'d':4.99,'e':12.57,'f':1.02,'g':1.30,
        'h':1.28,'i':6.18,'j':0.40,'k':0.02,'l':2.78,'m':4.74,'n':5.05,
        'o':10.73,'p':2.52,'q':1.20,'r':6.53,'s':7.81,'t':4.34,'u':4.63,
        'v':1.67,'w':0.01,'x':0.21,'y':0.01,'z':0.47}
prob_letra = np.array([freq[c] for c in ALFABETO])
prob_letra = prob_letra / prob_letra.sum()    # normaliza p/ somar 1

N = 500_000
idx = rng.choice(26, size=(N,5), p=prob_letra)         # sample com prob (== 'prob' do R)
seqs_d = np.array([''.join(ALFABETO[i] for i in linha) for linha in idx])
tem_a    = np.fromiter(('a' in s for s in seqs_d), bool, N)
valida_d = np.fromiter((s in dic5 for s in seqs_d), bool, N)
p_d_mc   = valida_d[tem_a].mean()

def prob_seq(w):
    p = 1.0
    for ch in w: p *= prob_letra[ord(ch)-97]
    return p
num = sum(prob_seq(w) for w in dic5 if 'a' in w)
P_A = 1 - (1-prob_letra[0])**5
p_d_teorico = num / P_A
print(f"P(válida | tem 'a')  MC        = {p_d_mc:.6f}")
print(f"P(válida | tem 'a')  analítico = {p_d_teorico:.6f}")

# ---- Gráfico (idêntico ao relatório) ----
nnd = np.arange(1, N + 1)
cv  = np.cumsum(valida_d & tem_a) / np.maximum(np.cumsum(tem_a), 1)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(nnd, cv, color=C3, lw=1, label="Estimativa MC  P(válida | ≥1 'a')")
ax.axhline(p_d_teorico, color=C2, ls="--", lw=1.6, label=f"Valor teórico = {p_d_teorico:.5f}")
ax.set_xscale("log")
ax.set_xlabel("Tamanho da amostra (log)"); ax.set_ylabel("Probabilidade")
ax.set_title("Gerador ponderado por frequência — P(válida | contém 'a')")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("graficos/Q1d_ponderado_cond_a.png", dpi=130, bbox_inches="tight")
plt.show()

# Questão 2 — Geração de variáveis aleatórias
## Item (a) — Cauchy pela transformada inversa
$F^{-1}(u)=\gamma\tan(\pi(u-1/2))$.

In [ ]:
def rcauchy_inversa(n, gamma=1.0, rng=None):
    """Amostra Cauchy(gamma) pelo método da transformada inversa."""
    rng = np.random.default_rng() if rng is None else rng
    u = rng.uniform(size=n)
    return gamma * np.tan(np.pi * (u - 0.5))

x = rcauchy_inversa(200_000, gamma=1.0, rng=rng)

# ---- Gráfico (idêntico ao relatório) ----
print("mediana:", np.median(x), "| semi-IQR:", (np.percentile(x,75)-np.percentile(x,25))/2)
xc = x[(x > -6) & (x < 6)]
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(xc, bins=120, density=True, color=C1, alpha=.55, label="Amostra (transf. inversa)")
t = np.linspace(-6, 6, 400)
ax.plot(t, 1/(np.pi*(1+t**2)), color=C2, lw=2, label="Densidade Cauchy(γ=1)")
ax.set_xlim(-6, 6); ax.set_xlabel("x"); ax.set_ylabel("densidade")
ax.set_title("Cauchy padrão via transformada inversa (recorte [-6,6])")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("graficos/Q2a_cauchy_inversa.png", dpi=130, bbox_inches="tight")
plt.show()

## Item (b) — Variável discreta pela transformada inversa

In [ ]:
valores = np.array([2, 3, 5, 6, 9])
probs   = np.array([0.3, 0.1, 0.1, 0.3, 0.2])
F_acum  = np.cumsum(probs)                       # [0.3,0.4,0.5,0.8,1.0]

def amostra_inversa_discreta(n, rng):
    u = rng.uniform(size=n)
    return valores[np.searchsorted(F_acum, u)]   # transformada inversa

n = 1000
amostra1 = amostra_inversa_discreta(n, rng)      # método da inversa
amostra2 = rng.choice(valores, size=n, p=probs)  # equivalente ao sample() do R

freq_rel1 = np.array([(amostra1 == v).mean() for v in valores])
freq_rel2 = np.array([(amostra2 == v).mean() for v in valores])
print("x  teórica  inversa  sample")
for v,p,e1,e2 in zip(valores,probs,freq_rel1,freq_rel2):
    print(f"{v}   {p:.2f}    {e1:.3f}    {e2:.3f}")

# ---- Gráfico (idêntico ao relatório) ----
fig, ax = plt.subplots(figsize=(7, 4))
w = 0.27; xp = np.arange(len(valores))
ax.bar(xp-w, probs, w, label="Teórica", color=C2)
ax.bar(xp,   freq_rel1, w, label="Empírica (inversa)", color=C1)
ax.bar(xp+w, freq_rel2, w, label="Empírica (sample)", color=C3)
ax.set_xticks(xp); ax.set_xticklabels(valores); ax.set_xlabel("x"); ax.set_ylabel("probabilidade")
ax.set_title("Frequências relativas (n=1000) vs teóricas")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("graficos/Q2b_discreta_inversa.png", dpi=130, bbox_inches="tight")
plt.show()

## Item (c) — Normal padrão por aceitação-rejeição (candidato Cauchy)
$M=\sup f/g=\sqrt{\pi/2}\,2e^{-1/2}\approx 1.520$; taxa de aceitação $=1/M$.

In [ ]:
def rnorm_aceitacao_rejeicao(n, rng):
    """Gera N(0,1) por aceitação-rejeição usando candidato Cauchy padrão."""
    M = np.sqrt(np.pi/2) * 2 * np.exp(-0.5)     # constante de cobertura
    aceitos = []
    while len(aceitos) < n:
        m = n - len(aceitos)
        y = rcauchy_inversa(m, gamma=1.0, rng=rng)        # candidato g ~ Cauchy
        u = rng.uniform(size=m)
        f = (1/np.sqrt(2*np.pi))*np.exp(-y**2/2)         # alvo  f ~ N(0,1)
        g = 1/(np.pi*(1+y**2))                           # proposta g
        aceitos.extend(y[u <= f/(M*g)].tolist())            # critério de aceitação
    return np.array(aceitos[:n])

# ---- Gráfico (idêntico ao relatório) ----
M = np.sqrt(np.pi/2) * 2 * np.exp(-0.5)          # constante de cobertura (sup f/g)
amostra_normal = rnorm_aceitacao_rejeicao(200_000, rng)
print(f"M={M:.4f} | média={amostra_normal.mean():.4f} | sd={amostra_normal.std():.4f}")
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(amostra_normal, bins=100, density=True, color=C1, alpha=.55, label="Amostra (aceit.-rejeição)")
t = np.linspace(-4, 4, 400)
ax.plot(t, 1/np.sqrt(2*np.pi)*np.exp(-t**2/2), color=C2, lw=2, label="N(0,1) teórica")
ax.set_xlim(-4.5, 4.5); ax.set_xlabel("x"); ax.set_ylabel("densidade")
ax.set_title("Normal padrão via aceitação-rejeição (candidato Cauchy)")
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig("graficos/Q2c_normal_aceit_rejeicao.png", dpi=130, bbox_inches="tight")
plt.show()